# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/areebahassann/Starter_Notebook/blob/main/work/notebooks/w03_data_contract.ipynb)

**Lane: Page/URL lane**  
**Author: Areeba Hassan**

In [ ]:
# Setup — install DuckDB
!pip install duckdb --quiet

import duckdb
import pandas as pd
import numpy as np

# Load CSV
import urllib.request
urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/areebahassann/Starter_Notebook/main/data/raw/content_refresh_anonymized.csv',
    'content_refresh_anonymized.csv'
)

# Register in DuckDB
con = duckdb.connect()
con.execute("CREATE TABLE pages AS SELECT * FROM read_csv_auto('content_refresh_anonymized.csv')")

# Quick check
total = con.execute('SELECT COUNT(*) FROM pages').fetchone()[0]
print(f'✅ Data loaded! Total rows: {total}')
con.execute('SELECT * FROM pages LIMIT 3').df()

## 1. Unit of analysis + time window

**One row = one content page (URL), observed as a single snapshot.**

- The grain is `content_id` — each row is one unique page.
- There is no `month` column — the dataset is a **point-in-time snapshot** of 30,000 pages.
- Metrics cover two windows already baked in: `_90d` (last 90 days) and `_last_30d` vs `_prev_30d` (for trend comparison).
- We use the full snapshot as our 'slice' and verify grain below.

In [ ]:
# Verify grain: content_id should be unique (no duplicates)
con.execute("""
    SELECT
        COUNT(*)                    AS total_rows,
        COUNT(DISTINCT content_id)  AS unique_content_ids,
        COUNT(*) - COUNT(DISTINCT content_id) AS duplicates
    FROM pages
""").df()

## 2. Fields: feature / label / context / excluded

### Label (what we predict)
| Field | Role | Notes |
|---|---|---|
| `needs_refresh` | **Label (derived)** | We define this: TRUE if `trend_direction='down'` AND `avg_position > 20`. A page is losing ground and ranks poorly — it needs a content update. |

### Features (inputs — all knowable BEFORE the decision)
| Field | Role | Why knowable |
|---|---|---|
| `avg_position` | Feature | Measured from past GSC data |
| `ctr` | Feature | Measured from past GSC data |
| `impressions_last_30d` | Feature | Accumulated over past 30 days |
| `content_age_days` | Feature | Age of content at decision time |
| `word_count` | Feature | Measured from current published page |

### Context (for filtering/grouping only)
| Field | Role |
|---|---|
| `content_id` | Unique page identifier |
| `client_id` | Which client owns the page |
| `trend_direction` | Used to DEFINE label — not a model input |
| `impression_tier`, `position_tier` | Bucketed versions of features — context only |

### Excluded
| Field | Why excluded |
|---|---|
| `trend_pct` | Partially derived from label logic — leakage risk |
| `model_used`, `provider_used` | Internal operational metadata, not predictive |
| `client_id` | Client-identifying — excluded per DATA_USE.md |

In [ ]:
# Show column names and types
con.execute('DESCRIBE pages').df()

## 3. Verify with queries (grain, counts, missing values, availability)

Every claim above gets a query. A contract claim without a query is a guess.

In [ ]:
# QUERY 1: Row count and basic stats
con.execute("""
    SELECT
        COUNT(*)                        AS total_pages,
        COUNT(DISTINCT content_id)      AS unique_pages,
        ROUND(AVG(avg_position), 2)     AS mean_position,
        ROUND(AVG(ctr), 4)              AS mean_ctr,
        ROUND(AVG(content_age_days), 1) AS mean_age_days
    FROM pages
""").df()

In [ ]:
# QUERY 2: Availability — pages that have GSC impression data (our 'available' proxy)
con.execute("""
    SELECT
        COUNT(*)  AS total_rows,
        SUM(CASE WHEN impressions_last_30d > 0 THEN 1 ELSE 0 END)  AS has_impressions,
        ROUND(
            100.0 * SUM(CASE WHEN impressions_last_30d > 0 THEN 1 ELSE 0 END) / COUNT(*), 1
        ) AS pct_with_impressions
    FROM pages
""").df()

In [ ]:
# QUERY 3: Missing value check on our 5 features
con.execute("""
    SELECT
        SUM(CASE WHEN avg_position          IS NULL THEN 1 ELSE 0 END) AS null_avg_position,
        SUM(CASE WHEN ctr                   IS NULL THEN 1 ELSE 0 END) AS null_ctr,
        SUM(CASE WHEN impressions_last_30d  IS NULL THEN 1 ELSE 0 END) AS null_impressions_30d,
        SUM(CASE WHEN content_age_days      IS NULL THEN 1 ELSE 0 END) AS null_age_days,
        SUM(CASE WHEN word_count            IS NULL THEN 1 ELSE 0 END) AS null_word_count
    FROM pages
""").df()

## 4. Data limits

What this data **cannot** tell us — honest limits:

1. **Snapshot only, not time series**: This is one point-in-time snapshot. We cannot observe how a page's ranking changed week by week — only where it stands now vs. the last 30/90 days.

2. **Label is a proxy, not ground truth**: `needs_refresh` is defined by us using `trend_direction` and `avg_position`. A human expert might disagree — this is a *directional* signal, not a verified label.

3. **GSC-only coverage**: Pages with zero impressions have no search signal. The model is blind to pages that Google never surfaced — these could be the most in need of refresh.

4. **Anonymization ceiling**: `client_id`, URLs, keywords, and titles are stripped. We cannot build topic-aware or niche-aware features — a tech article and a recipe look identical if their metrics match.

5. **Unbalanced history**: `_90d` metrics cover three months but `_last_30d` covers one. Seasonal pages (holiday content, etc.) may appear artificially weak in the shorter window.

> All results are **observed / measured / directional / decision-support** — not predictions of Google's algorithm.

## 5. Feature frame — 5 features for our lane

In [ ]:
# Build the feature frame with derived label
df = con.execute("""
    SELECT
        content_id,
        avg_position,           -- knowable: measured from past GSC data
        ctr,                    -- knowable: clicks/impressions from past GSC data
        impressions_last_30d,   -- knowable: accumulated over past 30 days
        content_age_days,       -- knowable: age of published content at decision time
        word_count,             -- knowable: measured from current published page
        -- Derive the label
        CASE
            WHEN trend_direction = 'down' AND avg_position > 20 THEN TRUE
            ELSE FALSE
        END AS needs_refresh
    FROM pages
    WHERE impressions_last_30d > 0
""").df()

print(f"Feature frame shape: {df.shape}")
print(f"\nLabel distribution:")
print(df['needs_refresh'].value_counts())
df.head(10)

## 6. The Leakage Trap

We add one label-derived column on purpose → score jumps toward perfect → delete it → keep the honest number.

> **Lesson**: A near-perfect score is a red flag, not a win.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score

# Drop any rows with nulls in our features
features = ['avg_position', 'ctr', 'impressions_last_30d', 'content_age_days', 'word_count']
df_clean = df[features + ['needs_refresh']].dropna()

X = df_clean[features]
y = df_clean['needs_refresh'].astype(int)

# STEP 1: Honest model
honest_score = cross_val_score(
    DecisionTreeClassifier(max_depth=4, random_state=42),
    X, y, cv=5, scoring='accuracy'
).mean()
print(f"✅ Honest model accuracy (5-fold CV): {honest_score:.3f}")

# STEP 2: Add leaky column (directly encodes the label!)
df_clean = df_clean.copy()
df_clean['leaky_feature'] = df_clean['needs_refresh'].astype(int) * 99
X_leaky = df_clean[features + ['leaky_feature']]
leaky_score = cross_val_score(
    DecisionTreeClassifier(max_depth=4, random_state=42),
    X_leaky, y, cv=5, scoring='accuracy'
).mean()
print(f"🚨 Leaky model accuracy (5-fold CV): {leaky_score:.3f}  ← suspiciously perfect!")

# STEP 3: Delete leaky column
df_clean.drop(columns=['leaky_feature'], inplace=True)
print(f"\n🗑️  Leaky column deleted.")
print(f"✅ Keeping honest score: {honest_score:.3f}")
print("\nLesson: High score on training data = model cheated. Honest score is what matters.")

## Self-check

- [x] Every section filled — markdown thinking AND code that backs it
- [x] Notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] Claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to repo under `work/notebooks/` — submit repo URL on the card